# Verify the AgentCore Infrastructure

The workshop-agentcore-stack is pre-deployed by Workshop Studio. This notebook
verifies the stack is healthy, captures outputs, and confirms key resources exist.

**What the stack provides:** Persona IAM roles, AgentCore Gateway with 3 Lambda targets,
request/response interceptors, WorkloadIdentity for M2M auth. Cognito and Lambda tools
are imported from the Registry and Tools Gateway stacks.

In [ ]:
%pip install "boto3==1.42.87" --quiet
import boto3
assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 42, 87), f"boto3 too old: {boto3.__version__} (need >=1.42.87)"
print(f"boto3 {boto3.__version__} OK")

In [ ]:
import boto3, json, time

region = boto3.session.Session().region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]
cfn = boto3.client("cloudformation", region_name=region)

STACK_NAME = "workshop-agentcore-stack"

print(f"Region:     {region}")
print(f"Account:    {account_id}")
print(f"Stack name: {STACK_NAME}")

## Step 1: Verify the Stack is Deployed

The stack is pre-provisioned by Workshop Studio. Confirm it is `CREATE_COMPLETE`.

In [ ]:
STACK_NAME = "workshop-agentcore-stack"

cfn = boto3.client("cloudformation", region_name=region)

try:
    stack = cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
    status = stack["StackStatus"]
    print(f"Stack: {STACK_NAME}")
    print(f"Status: {status}")
    print(f"Created: {stack['CreationTime']}")
    assert status == "CREATE_COMPLETE", f"Expected CREATE_COMPLETE, got {status}"
    print("\n✓ Stack is healthy")
except cfn.exceptions.ClientError as e:
    print(f"ERROR: {e}")
    print("The workshop-agentcore-stack must be deployed before running this notebook.")
    raise


## Step 2: Verify All Four Stacks

The AgentCore stack depends on the Registry and Tools Gateway stacks. Confirm all are healthy.

In [ ]:
REQUIRED_STACKS = [
    "workshop-llm-gateway-stack",
    "workshop-registry-stack",
    "workshop-tools-gateway-stack",
    "workshop-agentcore-stack",
]

missing_stacks = []
bad_status_stacks = []
for name in REQUIRED_STACKS:
    try:
        s = cfn.describe_stacks(StackName=name)["Stacks"][0]
        status = s["StackStatus"]
        print(f"{name}: {status}")
        if status not in ("CREATE_COMPLETE", "UPDATE_COMPLETE"):
            bad_status_stacks.append((name, status))
    except cfn.exceptions.ClientError:
        print(f"{name}: NOT FOUND")
        missing_stacks.append(name)

if missing_stacks:
    raise RuntimeError(
        f"Required stack(s) not found: {missing_stacks}. "
        f"Workshop Studio may not have finished provisioning — retry in 2 minutes."
    )
if bad_status_stacks:
    raise RuntimeError(
        f"Stacks not in a ready state: {bad_status_stacks}. "
        f"Check CloudFormation console for failure details."
    )

## Step 3: Verify Outputs

In [ ]:
stack = cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}

print(f"Stack: {STACK_NAME}")
print(f"Status: {stack['StackStatus']}")
print(f"\nOutputs ({len(outputs)}):")
print("=" * 70)
for key in sorted(outputs):
    val = outputs[key]
    if len(val) > 50:
        val = val[:47] + "..."
    print(f"  {key:35s} {val}")

## Step 4: Verify Key Resources

In [ ]:
# Verify Gateway is ready
agentcore = boto3.client("bedrock-agentcore-control", region_name=region)
gateways = agentcore.list_gateways().get("items", [])
gw = [g for g in gateways if g["name"] == "ac-tools-gateway"]
if gw:
    print(f"Gateway: {gw[0]['name']}")
    print(f"  Status: {gw[0]['status']}")
    print(f"  ID:     {gw[0]['gatewayId']}")

    # List targets
    targets = agentcore.list_gateway_targets(gatewayIdentifier=gw[0]["gatewayId"]).get("items", [])
    print(f"\nGateway Targets ({len(targets)}):")
    for t in targets:
        print(f"  - {t['name']} ({t['status']})")
else:
    raise RuntimeError(
        "Gateway 'ac-tools-gateway' not found. Deploy (or re-deploy) the "
        "workshop-agentcore-stack (Module 3b) before continuing. If you ran Module 4 "
        "first and it created 'tools-gateway' instead, update the Module 3b stack."
    )

# Verify Lambda functions
lam = boto3.client("lambda", region_name=region)
missing_lambdas = []
for fn_name in ["workshop-flights-mcp", "workshop-hotels-mcp", "workshop-search-knowledge-base",
                "workshop-product-info-tool", "workshop-order-processing-agent"]:
    try:
        fn = lam.get_function(FunctionName=fn_name)
        print(f"\nLambda: {fn_name} \u2713")
    except lam.exceptions.ResourceNotFoundException:
        print(f"\nLambda: {fn_name} \u2717 NOT FOUND")
        missing_lambdas.append(fn_name)

if missing_lambdas:
    raise RuntimeError(
        f"Required Lambda function(s) missing: {missing_lambdas}. "
        f"Re-deploy workshop-agentcore-stack to provision them."
    )

# Verify Cognito — the User Pool ID lives on the Registry stack (data-stack),
# imported by workshop-agentcore-stack via CloudFormation exports.
cognito = boto3.client("cognito-idp", region_name=region)
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]
pool_id = exports.get("workshop-CognitoUserPoolId", "")
if pool_id:
    pool = cognito.describe_user_pool(UserPoolId=pool_id)["UserPool"]
    print(f"\nCognito User Pool: {pool['Name']} ({pool_id})")
else:
    print("\nCognito User Pool: export workshop-CognitoUserPoolId not found")

## Summary

| Resource Category | Count | Source Stack |
|---|---|---|
| Cognito Identity | 12 | Registry Stack (shared) |
| Persona IAM Roles | 3 | AgentCore Stack |
| Lambda Tools | 5 | Tools Gateway Stack (shared) |
| AgentCore Gateway + Targets | 4 | AgentCore Stack |
| Interceptors | 2 | AgentCore Stack |
| WorkloadIdentity | 1 | AgentCore Stack |

The infrastructure is ready. In the next notebook, you'll create the AgentCore Registry
and start registering tools.